# MRI脳腫瘍画像の小摂動Adversarial Attack検知（実験2）

実験1のMobileNetV2最終特徴fine-tuningは比較baselineとして保存し、このNotebookでは独立した新規実験を行う。目的は、小さいFGSM摂動のうち、元の腫瘍分類モデルの正しい診断を誤診に変える攻撃を低い誤検知率で検出することである。

- 検知特徴：分類確率、最終内部特徴、軽いblur前後の一貫性
- 学習positive：cleanで正解し、FGSM後に誤分類した攻撃
- 学習negative：clean画像
- 閾値：ValidationでFPR 10%以下となる点から選択
- Testing：閾値とモデルの確定後のみ使用


In [ ]:
import json
import os
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    precision_score, recall_score, roc_auc_score, roc_curve,
)

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.20
TRAIN_EPSILONS = (0.01, 0.1, 0.5)
UNSEEN_EPSILONS = (0.05, 0.25, 1.0)
ALL_EVALUATION_EPSILONS = tuple(sorted(
    set(TRAIN_EPSILONS + UNSEEN_EPSILONS)
))
TARGET_MAX_FPR = 0.10

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## Step 1. Drive、データ、分類モデルの準備


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ThammasatResearch")
DATASET_ZIP_PATH = DRIVE_ROOT / "dataset" / "archive.zip"
EXTRACT_ROOT = Path("/content/brain_tumor")
TRAINING_DIRECTORY = EXTRACT_ROOT / "Training"
TESTING_DIRECTORY = EXTRACT_ROOT / "Testing"
CLASSIFIER_MODEL_PATH = DRIVE_ROOT / "models" / "baseline_mobilenetv2.keras"
V2_RESULTS_DIRECTORY = DRIVE_ROOT / "results" / "detection_v2"
V2_MODELS_DIRECTORY = DRIVE_ROOT / "models" / "detection_v2"
V2_RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
V2_MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)
assert DATASET_ZIP_PATH.exists(), DATASET_ZIP_PATH
assert CLASSIFIER_MODEL_PATH.exists(), CLASSIFIER_MODEL_PATH

if not (TRAINING_DIRECTORY.exists() and TESTING_DIRECTORY.exists()):
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zip_file:
        zip_file.extractall(EXTRACT_ROOT)
print("Results:", V2_RESULTS_DIRECTORY)


In [ ]:
train_dataset, validation_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAINING_DIRECTORY, validation_split=VALIDATION_SPLIT, subset="both",
    seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=True,
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TESTING_DIRECTORY, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    label_mode="int", shuffle=False,
)
train_paths = set(train_dataset.file_paths)
validation_paths = set(validation_dataset.file_paths)
assert not (train_paths & validation_paths)
assert len(train_paths) == 4480 and len(validation_paths) == 1120

classifier_model = tf.keras.models.load_model(CLASSIFIER_MODEL_PATH)
classifier_model.trainable = False
gap_layers = [
    layer for layer in classifier_model.layers
    if isinstance(layer, tf.keras.layers.GlobalAveragePooling2D)
]
if not gap_layers:
    raise RuntimeError("GlobalAveragePooling2D layer was not found.")
representation_model = tf.keras.Model(
    classifier_model.input,
    [gap_layers[-1].output, classifier_model.output],
    name="classifier_representation_model",
)
representation_model.trainable = False
print("Classes:", train_dataset.class_names)
print("Representation:", gap_layers[-1].name, gap_layers[-1].output.shape)


## Step 2. FGSMと検知特徴の生成

blurは攻撃を除去する防御としてではなく、分類器の出力と内部表現の一貫性を測るために使用する。


In [ ]:
classification_loss = tf.keras.losses.SparseCategoricalCrossentropy()

@tf.function
def create_fgsm(images, labels, epsilon):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.int32)
    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = classifier_model(images, training=False)
        loss = classification_loss(labels, predictions)
    gradients = tape.gradient(loss, images)
    adversarial = tf.clip_by_value(
        images + tf.cast(epsilon, tf.float32) * tf.sign(gradients),
        0.0, 255.0,
    )
    return predictions, adversarial


@tf.function
def extract_detection_features(images):
    images = tf.cast(images, tf.float32)
    blurred = tf.nn.avg_pool2d(images, ksize=3, strides=1, padding="SAME")
    original_features, original_probabilities = representation_model(
        images, training=False
    )
    blurred_features, blurred_probabilities = representation_model(
        blurred, training=False
    )
    probability_difference = tf.abs(
        original_probabilities - blurred_probabilities
    )
    feature_difference = tf.abs(original_features - blurred_features)
    confidence = tf.reduce_max(original_probabilities, axis=1, keepdims=True)
    sorted_probabilities = tf.sort(original_probabilities, axis=1)
    margin = (sorted_probabilities[:, -1] - sorted_probabilities[:, -2])[:, None]
    entropy = -tf.reduce_sum(
        original_probabilities * tf.math.log(original_probabilities + 1e-7),
        axis=1, keepdims=True,
    )
    difference_summary = tf.stack([
        tf.reduce_mean(feature_difference, axis=1),
        tf.math.reduce_std(feature_difference, axis=1),
        tf.reduce_max(feature_difference, axis=1),
    ], axis=1)
    return tf.concat([
        original_features, original_probabilities, blurred_probabilities,
        probability_difference, confidence, margin, entropy, difference_summary,
    ], axis=1)

sample_images, sample_labels = next(iter(train_dataset))
sample_features = extract_detection_features(sample_images)
print("Detection feature dimension:", sample_features.shape[1])


In [ ]:
def collect_split_features(source_dataset, epsilons, split_name):
    clean_feature_batches = []
    clean_correct_batches = []
    adversarial_records = {epsilon: {"features": [], "success": []}
                           for epsilon in epsilons}

    for batch_index, (images, labels) in enumerate(source_dataset):
        labels_np = labels.numpy().reshape(-1)
        clean_features = extract_detection_features(images).numpy()
        clean_probabilities = classifier_model(images, training=False).numpy()
        clean_classes = np.argmax(clean_probabilities, axis=1)
        clean_correct = clean_classes == labels_np
        clean_feature_batches.append(clean_features)
        clean_correct_batches.append(clean_correct)

        for epsilon in epsilons:
            _, adversarial_images = create_fgsm(images, labels, epsilon)
            adversarial_features = extract_detection_features(
                adversarial_images
            ).numpy()
            adversarial_probabilities = classifier_model(
                adversarial_images, training=False
            ).numpy()
            adversarial_classes = np.argmax(adversarial_probabilities, axis=1)
            successful = clean_correct & (adversarial_classes != labels_np)
            adversarial_records[epsilon]["features"].append(adversarial_features)
            adversarial_records[epsilon]["success"].append(successful)

        if (batch_index + 1) % 25 == 0:
            print(f"{split_name}: {batch_index + 1} batches")

    result = {
        "clean_features": np.concatenate(clean_feature_batches),
        "clean_correct": np.concatenate(clean_correct_batches),
        "adversarial": {},
    }
    for epsilon, record in adversarial_records.items():
        result["adversarial"][epsilon] = {
            "features": np.concatenate(record["features"]),
            "success": np.concatenate(record["success"]),
        }
    return result


train_feature_data = collect_split_features(
    train_dataset, TRAIN_EPSILONS, "Training"
)
validation_feature_data = collect_split_features(
    validation_dataset, TRAIN_EPSILONS, "Validation"
)


## Step 3. 成功攻撃検知器の新規学習


In [ ]:
def build_successful_attack_dataset(feature_data):
    negative_features = feature_data["clean_features"]
    positive_features = []
    counts = {}
    for epsilon in TRAIN_EPSILONS:
        record = feature_data["adversarial"][epsilon]
        selected = record["features"][record["success"]]
        positive_features.append(selected)
        counts[str(epsilon)] = int(selected.shape[0])
    positive_features = np.concatenate(positive_features)
    features = np.concatenate([negative_features, positive_features]).astype(
        np.float32
    )
    labels = np.concatenate([
        np.zeros(negative_features.shape[0], dtype=np.float32),
        np.ones(positive_features.shape[0], dtype=np.float32),
    ])
    return features, labels, counts

train_x, train_y, train_success_counts = build_successful_attack_dataset(
    train_feature_data
)
validation_x, validation_y, validation_success_counts = (
    build_successful_attack_dataset(validation_feature_data)
)
print("Training shape:", train_x.shape, train_success_counts)
print("Validation shape:", validation_x.shape, validation_success_counts)


In [ ]:
RUN_NAME = "successful_fgsm_consistency_detector"
MODEL_PATH = V2_MODELS_DIRECTORY / f"{RUN_NAME}.keras"
HISTORY_PATH = V2_RESULTS_DIRECTORY / f"{RUN_NAME}_history.csv"
CONFIG_PATH = V2_RESULTS_DIRECTORY / f"{RUN_NAME}_config.json"
FORCE_RETRAIN = False

normalizer = tf.keras.layers.Normalization(name="feature_normalization")
normalizer.adapt(train_x)
inputs = tf.keras.Input(shape=(train_x.shape[1],), name="detection_features")
hidden = normalizer(inputs)
hidden = tf.keras.layers.Dense(256, activation="relu")(hidden)
hidden = tf.keras.layers.Dropout(0.35)(hidden)
hidden = tf.keras.layers.Dense(64, activation="relu")(hidden)
hidden = tf.keras.layers.Dropout(0.20)(hidden)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(hidden)
detector = tf.keras.Model(inputs, outputs, name=RUN_NAME)
detector.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="roc_auc"),
             tf.keras.metrics.AUC(name="pr_auc", curve="PR")],
)
negative_count = float(np.sum(train_y == 0))
positive_count = float(np.sum(train_y == 1))
class_weight = {0: 1.0, 1: negative_count / positive_count}
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_roc_auc", mode="max", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc", mode="max", patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc", mode="max", patience=2, factor=0.5,
        min_lr=1e-6
    ),
]

if FORCE_RETRAIN or not (MODEL_PATH.exists() and HISTORY_PATH.exists()):
    history = detector.fit(
        train_x, train_y, validation_data=(validation_x, validation_y),
        epochs=30, batch_size=64, class_weight=class_weight,
        callbacks=callbacks, verbose=1,
    )
    pd.DataFrame(history.history).to_csv(HISTORY_PATH, index_label="epoch")
else:
    print("Using saved experiment-2 detector.")
detector = tf.keras.models.load_model(MODEL_PATH)
json.dump({
    "run_name": RUN_NAME, "seed": SEED,
    "training_epsilons": TRAIN_EPSILONS,
    "positive_definition": "clean correct and adversarial incorrect",
    "target_max_fpr": TARGET_MAX_FPR,
    "train_success_counts": train_success_counts,
    "validation_success_counts": validation_success_counts,
}, CONFIG_PATH.open("w", encoding="utf-8"), indent=2)


## Step 4. ValidationでFPR制約付き閾値を選択

Testingデータはこの閾値選択に使用しない。


In [ ]:
validation_scores = detector.predict(validation_x, batch_size=256).reshape(-1)
fpr_values, tpr_values, threshold_values = roc_curve(
    validation_y, validation_scores
)
eligible = np.flatnonzero(fpr_values <= TARGET_MAX_FPR)
if eligible.size == 0:
    raise RuntimeError("No threshold satisfies the FPR constraint.")
best_index = eligible[np.argmax(tpr_values[eligible])]
SELECTED_THRESHOLD = float(threshold_values[best_index])
threshold_record = {
    "threshold": SELECTED_THRESHOLD,
    "validation_fpr": float(fpr_values[best_index]),
    "validation_successful_attack_tpr": float(tpr_values[best_index]),
    "validation_roc_auc": float(roc_auc_score(validation_y, validation_scores)),
    "validation_pr_auc": float(average_precision_score(
        validation_y, validation_scores
    )),
}
THRESHOLD_PATH = V2_RESULTS_DIRECTORY / f"{RUN_NAME}_threshold.json"
THRESHOLD_PATH.write_text(
    json.dumps(threshold_record, indent=2), encoding="utf-8"
)
print(json.dumps(threshold_record, indent=2))


## Step 5. 固定済みモデルと閾値によるTesting最終評価


In [ ]:
test_feature_data = collect_split_features(
    test_dataset, ALL_EVALUATION_EPSILONS, "Testing"
)
test_clean_scores = detector.predict(
    test_feature_data["clean_features"], batch_size=256
).reshape(-1)
test_clean_predictions = test_clean_scores >= SELECTED_THRESHOLD
test_fpr = float(np.mean(test_clean_predictions))


In [ ]:
test_rows = []
for epsilon in ALL_EVALUATION_EPSILONS:
    record = test_feature_data["adversarial"][epsilon]
    adversarial_scores = detector.predict(
        record["features"], batch_size=256
    ).reshape(-1)
    successful = record["success"]
    successful_scores = adversarial_scores[successful]
    combined_labels = np.concatenate([
        np.zeros(test_clean_scores.size),
        np.ones(successful_scores.size),
    ])
    combined_scores = np.concatenate([test_clean_scores, successful_scores])
    initially_correct = test_feature_data["clean_correct"]
    initially_correct_count = int(np.sum(initially_correct))
    successful_count = int(np.sum(successful))
    row = {
        "epsilon": epsilon,
        "epsilon_status": "known" if epsilon in TRAIN_EPSILONS else "unseen",
        "source_images": int(successful.size),
        "initially_correct_images": initially_correct_count,
        "successful_attacks": successful_count,
        "attack_success_rate_on_initially_correct": (
            float(successful_count / initially_correct_count)
            if initially_correct_count else np.nan
        ),
        "clean_false_positive_rate": test_fpr,
        "successful_attack_detection_rate": (
            float(np.mean(successful_scores >= SELECTED_THRESHOLD))
            if successful_scores.size else np.nan
        ),
        "all_attack_detection_rate": float(np.mean(
            adversarial_scores >= SELECTED_THRESHOLD
        )),
        "successful_vs_clean_roc_auc": (
            float(roc_auc_score(combined_labels, combined_scores))
            if successful_scores.size else np.nan
        ),
        "successful_vs_clean_pr_auc": (
            float(average_precision_score(combined_labels, combined_scores))
            if successful_scores.size else np.nan
        ),
        "selected_threshold": SELECTED_THRESHOLD,
    }
    test_rows.append(row)

test_results = pd.DataFrame(test_rows).sort_values("epsilon")
TEST_RESULTS_PATH = V2_RESULTS_DIRECTORY / f"{RUN_NAME}_test_by_epsilon.csv"
TEST_PLOT_PATH = V2_RESULTS_DIRECTORY / f"{RUN_NAME}_test_by_epsilon.png"
test_results.to_csv(TEST_RESULTS_PATH, index=False)
display(test_results.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(test_results["epsilon"],
             test_results["attack_success_rate_on_initially_correct"],
             marker="o", label="Attack success rate")
axes[0].plot(test_results["epsilon"],
             test_results["successful_attack_detection_rate"],
             marker="o", label="Successful attack detection rate")
axes[0].axhline(0.8, color="gray", linestyle="--", label="80% target")
axes[0].set(xlabel="Epsilon (0-255 scale)", ylabel="Rate", ylim=(0, 1),
            title="Harmful attack detection")
axes[0].grid(True); axes[0].legend()
axes[1].plot(test_results["epsilon"],
             test_results["successful_vs_clean_roc_auc"],
             marker="o", label="ROC-AUC")
axes[1].plot(test_results["epsilon"],
             test_results["successful_vs_clean_pr_auc"],
             marker="o", label="PR-AUC")
axes[1].axhline(0.5, color="gray", linestyle="--")
axes[1].set(xlabel="Epsilon (0-255 scale)", ylabel="Metric", ylim=(0, 1),
            title=f"Successful attacks vs clean (test FPR={test_fpr:.3f})")
axes[1].grid(True); axes[1].legend()
plt.tight_layout()
plt.savefig(TEST_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", TEST_RESULTS_PATH)
print("Saved:", TEST_PLOT_PATH)
print("Experiment 2 completed.")


## 判定基準

Testingで`clean_false_positive_rate <= 0.10`かつ、小さいεの`successful_attack_detection_rate >= 0.80`を主目標とする。未達の場合はTesting結果に合わせてこのモデルを再調整せず、新しい実験として検出特徴や攻撃を追加する。
